# 03 — Grid Snapping and the Feistel PRP

## Act 1: Grid Snapping

After projecting (lat, lon) to (x, y) in metres, we quantise to the nearest 250 m grid tile:

```
qx = round(x / bin)     qy = round(y / bin)
rx = x - qx * bin       ry = y - qy * bin
```

The residual `(rx, ry)` is always in **(-125, +125) m** — the full sub-tile precision of the original GPS fix. Storing `(qx, qy)` in plain text would reveal the 250 m tile; that is why we need to shuffle it.

## Act 2 Preview: The Permutation Requirement

We need to map every tile index `(qx, qy)` to an encrypted index `(qxp, qyp)` such that:
1. No two tiles map to the same encrypted tile (injectivity).
2. The mapping is reversible (surjectivity over the same finite set).
3. The mapping is keyed and computationally indistinguishable from random.

A keyed bijection on a finite set is a **pseudorandom permutation (PRP)**. The Feistel network construction guarantees this.

In [1]:
import math
import struct
import hashlib
import secrets
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from map_encryption import (
    _project, _unproject, _grid_range, _prp_encrypt, _prp_decrypt,
    _prf_upto, SchemeParams, _R_EARTH,
)

def metres_to_deg(spread_m, at_lat):
    lat_deg = spread_m / 111_320
    lon_deg = spread_m / (111_320 * math.cos(math.radians(at_lat)))
    return lat_deg, lon_deg

params = SchemeParams()
BIN = params.bin_size_m
CENTER_LAT, CENTER_LON = 40.758, -73.985  # Times Square

In [2]:
x_ts, y_ts = _project(CENTER_LAT, CENTER_LON)
qx = int(round(x_ts / BIN))
qy = int(round(y_ts / BIN))
rx = x_ts - qx * BIN
ry = y_ts - qy * BIN

print(f'Times Square projection:')
print(f'  x = {x_ts:.3f} m,  y = {y_ts:.3f} m')
print(f'  qx = {qx},  qy = {qy}')
print(f'  rx = {rx:.4f} m,  ry = {ry:.4f} m')

assert abs(rx) < 125, f'rx={rx} outside tile half-width'
assert abs(ry) < 125, f'ry={ry} outside tile half-width'
print('Residual is within tile half-width: OK')

Times Square projection:
  x = -8235972.526 m,  y = 4976711.982 m
  qx = -32944,  qy = 19907
  rx = 27.4737 m,  ry = -38.0177 m
Residual is within tile half-width: OK


In [3]:
min_q, M = _grid_range(BIN)
print(f'Grid range: min_q = {min_q},  M = {M}')
print(f'Total cells: M^2 = {M**2:,}')

print('\n5x5 grid of (qx, qy) tiles near Times Square and their back-projected coords:')
print(f'{"(qx,qy)":<20} {"lat":>10} {"lon":>12}')
for dqy in range(-2, 3):
    for dqx in range(-2, 3):
        tqx, tqy = qx + dqx, qy + dqy
        tlat, tlon = _unproject(tqx * BIN, tqy * BIN)
        print(f'  ({tqx},{tqy})            {tlat:>10.6f} {tlon:>12.6f}')

Grid range: min_q = -80151,  M = 160302
Total cells: M^2 = 25,696,731,204

5x5 grid of (qx, qy) tiles near Times Square and their back-projected coords:
(qx,qy)                     lat          lon
  (-32946,19905)             40.754856   -73.989738
  (-32945,19905)             40.754856   -73.987493
  (-32944,19905)             40.754856   -73.985247
  (-32943,19905)             40.754856   -73.983001
  (-32942,19905)             40.754856   -73.980755
  (-32946,19906)             40.756558   -73.989738
  (-32945,19906)             40.756558   -73.987493
  (-32944,19906)             40.756558   -73.985247
  (-32943,19906)             40.756558   -73.983001
  (-32942,19906)             40.756558   -73.980755
  (-32946,19907)             40.758259   -73.989738
  (-32945,19907)             40.758259   -73.987493
  (-32944,19907)             40.758259   -73.985247
  (-32943,19907)             40.758259   -73.983001
  (-32942,19907)             40.758259   -73.980755
  (-32946,19908)      

## Why the Mapping Must Be a Bijection

If two tiles `(qx1, qy1)` and `(qx2, qy2)` mapped to the same encrypted tile `(qxp, qyp)`, decryption would be ambiguous — we would not know which original tile to recover. A PRP avoids this by construction: it is a **bijection** (one-to-one and onto) on the tile index set.

**Feistel round structure** (one round):
```
L_new = (L + F(R, key, tweak, round)) mod M
(L, R) = (R, L_new)    # swap
```
Regardless of `F`, reversing the swap and then subtracting `F(R, ...)` recovers the original. Even a non-invertible `F` produces an invertible round — that is the Feistel trick. After ≥ 4 rounds the composition is a PRP (Luby & Rackoff, 1988).

## Rejection Sampling for Unbiased PRF Output

Naive `hash % M` is **biased** when `2^256 mod M ≠ 0`. Suppose `2^256 = k*M + r` with `r > 0`. Values in `[0, r)` are hit `k+1` times; values in `[r, M)` only `k` times. Over millions of records, this creates a **frequency fingerprint**: an attacker observing encrypted tile indices can distinguish slightly-more-frequent values from slightly-less-frequent ones.

**Rejection sampling** fixes this:
```
cutoff = floor(2^256 / M) * M
loop:
    h = BLAKE2s(key, data + counter)
    val = int.from_bytes(h, 'big')
    if val < cutoff: return val % M
    counter += 1
```
Rejection probability `< M / 2^256 ≈ 6×10⁻⁷²` — virtually impossible. The result is exactly uniform over `[0, M)`.

In [4]:
min_q, M = _grid_range(BIN)
fixed_key = hashlib.blake2s(b'nb03-prf-demo', digest_size=32).digest()

cutoff = ((1 << 256) // M) * M
print(f'M = {M},  cutoff = {cutoff}')
print()

for i in range(3):
    data = struct.pack('>I', i)
    h = hashlib.blake2s(key=fixed_key, digest_size=32)
    h.update(data + struct.pack('>I', 0))
    raw = int.from_bytes(h.digest(), 'big')
    result = _prf_upto(fixed_key, data, M)
    accepted = raw < cutoff
    print(f'input={i}: raw_hash={raw} (accepted={accepted}), result={result}')
    assert 0 <= result < M, f'result {result} out of range [0, {M})'

print('\nAll results in [0, M): OK')

M = 160302,  cutoff = 115792089237316195423570985008687907853269984665640564039457584007913129538052

input=0: raw_hash=62063294506474534406807778697367844746765763659134447517792861568967532039976 (accepted=True), result=81864
input=1: raw_hash=95845104314074709888036525528181289095128042645880492658927600999069123810873 (accepted=True), result=78097
input=2: raw_hash=76660543974569812697702030828215209556824024407919041361775988918881059455720 (accepted=True), result=51776

All results in [0, M): OK


In [5]:
test_key = hashlib.blake2s(b'nb03-test', digest_size=32).digest()
test_tweak = b'nb03-demo'
min_q, M = _grid_range(BIN)

# Manual Feistel trace (4 rounds)
L, Rv = qx - min_q, qy - min_q
print(f'Initial (L, R) = ({L}, {Rv})')
for r in range(4):
    F = _prf_upto(test_key, test_tweak + struct.pack('>II', r, Rv), M)
    L = (L + F) % M
    L, Rv = Rv, L
    print(f'After round {r}: F={F}, (L, R) = ({L}, {Rv})')

# Undo the implicit final swap for 4 rounds (even number)
manual_qxp = L + min_q
manual_qyp = Rv + min_q

# Verify against _prp_encrypt (using 4 rounds via params override)
lib_qxp, lib_qyp = _prp_encrypt(qx, qy, test_key, test_tweak, BIN, 4)
print(f'\nManual result:  qxp={manual_qxp}, qyp={manual_qyp}')
print(f'Library result: qxp={lib_qxp}, qyp={lib_qyp}')
assert (manual_qxp, manual_qyp) == (lib_qxp, lib_qyp), 'Mismatch!'
print('Manual Feistel trace matches _prp_encrypt: OK')

Initial (L, R) = (47207, 100058)
After round 0: F=102977, (L, R) = (100058, 150184)
After round 1: F=8223, (L, R) = (150184, 108281)
After round 2: F=73047, (L, R) = (108281, 62929)
After round 3: F=33902, (L, R) = (62929, 142183)

Manual result:  qxp=-17222, qyp=62032
Library result: qxp=-17222, qyp=62032
Manual Feistel trace matches _prp_encrypt: OK


In [6]:
# PRP scatter: 25x25 = 625 tiles
prp_key = hashlib.blake2s(b'nb03-scatter', digest_size=32).digest()
prp_tweak = b'nb03-demo'

orig_qx, orig_qy, enc_qxp, enc_qyp = [], [], [], []
for dqx in range(-12, 13):
    for dqy in range(-12, 13):
        tqx, tqy = qx + dqx, qy + dqy
        tqxp, tqyp = _prp_encrypt(tqx, tqy, prp_key, prp_tweak, BIN, 10)
        orig_qx.append(tqx); orig_qy.append(tqy)
        enc_qxp.append(tqxp); enc_qyp.append(tqyp)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Original tile indices', 'Encrypted tile indices (PRP)'])
fig.add_trace(
    go.Scatter(x=orig_qx, y=orig_qy, mode='markers',
               marker=dict(color='steelblue', size=6), name='original'),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=enc_qxp, y=enc_qyp, mode='markers',
               marker=dict(color='tomato', size=6), name='encrypted'),
    row=1, col=2
)
fig.update_layout(
    title='Feistel PRP: 625 tiles before and after shuffling',
    template='plotly_white',
    height=400
)
fig.show()

In [7]:
import secrets
import pandas as pd
import folium

rng = np.random.default_rng(42)
n = 100
spread_m = 1500
dlat, dlon = metres_to_deg(spread_m, CENTER_LAT)

snap_key = secrets.token_bytes(32)
lats_s = CENTER_LAT + rng.uniform(-dlat, dlat, n)
lons_s = CENTER_LON + rng.uniform(-dlon, dlon, n)

rows = []
for i in range(n):
    x_i, y_i = _project(lats_s[i], lons_s[i])
    tqx = int(round(x_i / BIN))
    tqy = int(round(y_i / BIN))
    snap_lat, snap_lon = _unproject(tqx * BIN, tqy * BIN)
    rows.append({'lat': lats_s[i], 'lon': lons_s[i], 'type': 'original'})
    rows.append({'lat': snap_lat,  'lon': snap_lon,   'type': 'snapped (tile centre)'})

df_snap = pd.DataFrame(rows)

colors = {'original': 'steelblue', 'snapped (tile centre)': 'orange'}
m = folium.Map(location=[CENTER_LAT, CENTER_LON], zoom_start=14, tiles='OpenStreetMap')
for _, row in df_snap.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=5,
        color=colors[row['type']],
        fill=True,
        fill_color=colors[row['type']],
        fill_opacity=0.7,
        tooltip=row['type'],
    ).add_to(m)
m

## What Snapping + PRP Protect — and Don't

**Protected by snapping:** GPS precision is coarsened to ±125 m; even knowing (qx, qy), an adversary cannot determine the exact position within the tile.

**Protected by PRP:** Without the PRP key, (qxp, qyp) reveals no information about which tile was originally visited — including which country or continent.

**Not yet protected:** The residual `(rx, ry)` is still in plaintext at this stage. It carries the full within-tile precision of the original GPS fix. NB04 shows how AEAD encryption seals this final piece of information.